In [1]:
import torch
from tinytrie import TrieNode

def build_module_hierarchy_trie(module) :
    root = TrieNode()
    root.is_end = True
    root.value = module
    for child_module_name, child_module in module.named_children() :
        child_root = build_module_hierarchy_trie(child_module)
        root.children[child_module_name] = child_root
    return root

In [2]:
from typing import Dict
import torch
from tinytrie import TrieNode

def analyze_node_heights(module_hierarchy_trie_root):
    node_heights = {}
    def analyze_node_height(node):
        if node.children:
            children_heights = []
            for child_node in node.children.values():
                analyze_node_height(child_node)
                children_heights.append(node_heights[child_node])
                node_heights[node] = max(children_heights) + 1
        else:
            node_heights[node] = 0
    analyze_node_height(module_hierarchy_trie_root)
    return node_heights

In [3]:
from collections import Counter


def analyze_child_modules(modules):
    child_module_names_and_types_counter = Counter()
    for module in modules:
        for child_module_name, child_module in module.named_children():
            child_module_names_and_types_counter[(child_module_name, type(child_module))] += 1
    return child_module_names_and_types_counter


def analyze_parameters(modules):
    parameter_names_require_grads_shapes_counter = Counter()
    for module in modules:
        for parameter_name, parameter in module.named_parameters(recurse=False):
            parameter_names_require_grads_shapes_counter[(parameter_name, parameter.requires_grad, parameter.shape)] += 1
    return parameter_names_require_grads_shapes_counter


def analyze_buffers(modules):
    buffer_names_is_persistents_shapes_counter = Counter()
    for module in modules:
        state_dict = module.state_dict()
        for buffer_name, buffer in module.named_buffers(recurse=False):
            is_persistent = buffer_name in state_dict
            buffer_names_is_persistents_shapes_counter[(buffer_name, is_persistent, buffer.shape)] += 1
    return buffer_names_is_persistents_shapes_counter


def analyze_forwards(modules):
    forwards = Counter()
    for module in modules:
        forward = module.forward
        forwards[forward.__func__] += 1
    return forwards

------

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-30B-A3B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/root/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 531/531 [01:29<00:00,  


In [5]:
model

Qwen3MoeForCausalLM(
  (model): Qwen3MoeModel(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-47): 48 x Qwen3MoeDecoderLayer(
        (self_attn): Qwen3MoeAttention(
          (q_proj): Linear(in_features=2048, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2048, bias=False)
          (q_norm): Qwen3MoeRMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3MoeRMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MoeSparseMoeBlock(
          (experts): Qwen3MoeExperts(
            (act_fn): SiLUActivation()
          )
          (gate): Qwen3MoeTopKRouter()
        )
        (input_layernorm): Qwen3MoeRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen3MoeRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen3MoeRMSNorm((2048,), eps=1e-06)
    (r

In [6]:
module_hierarchy_trie_root = build_module_hierarchy_trie(model)

In [7]:
from tinytrie import collect_sequences

module_types_to_modules = {}
module_types_to_names = {}
module_types_to_nodes = {}

for sequence, node in collect_sequences(module_hierarchy_trie_root):
    name = '.'.join(sequence)
    module = node.value
    module_type = type(module)

    module_types_to_modules.setdefault(module_type, []).append(module)
    module_types_to_names.setdefault(module_type, []).append(name)
    module_types_to_nodes.setdefault(module_type, []).append(node)

In [8]:
node_heights = analyze_node_heights(module_hierarchy_trie_root)

In [9]:
module_types_to_heights = {
    module_type: max(node_heights[node] for node in nodes)
    for module_type, nodes in module_types_to_nodes.items()
}

In [10]:
user_defined_module_types_by_height = sorted((t for t in module_types_to_heights.keys() if not t.__module__.startswith('torch')), key=lambda t: module_types_to_heights[t])
user_defined_module_types_by_height

[transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRMSNorm,
 transformers.activations.SiLUActivation,
 transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeTopKRouter,
 transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRotaryEmbedding,
 transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeAttention,
 transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeExperts,
 transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeSparseMoeBlock,
 transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeDecoderLayer,
 transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeModel,
 transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeForCausalLM]

------

In [11]:
import inspect

In [12]:
import sys

In [13]:
def retrieve_global_name(module_type, global_name):
    return getattr(sys.modules[module_type.__module__], global_name)

------

In [14]:
module_type = user_defined_module_types_by_height[0]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRMSNorm

In [15]:
len(module_types_to_modules[module_type])

193

In [16]:
analyze_child_modules(module_types_to_modules[module_type])

Counter()

In [17]:
analyze_parameters(module_types_to_modules[module_type])

Counter({('weight', True, torch.Size([2048])): 97,
         ('weight', True, torch.Size([128])): 96})

In [18]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [19]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRMSNorm.forward(self, hidden_states: torch.Tensor) -> torch.Tensor>: 193})

In [20]:
print(inspect.getsource(next(iter(forwards))))

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)



In [21]:
{self.variance_epsilon for self in module_types_to_modules[module_type]}

{1e-06}

------

In [22]:
module_type = user_defined_module_types_by_height[1]
module_type

transformers.activations.SiLUActivation

In [23]:
len(module_types_to_modules[module_type])

48

In [24]:
analyze_child_modules(module_types_to_modules[module_type])

Counter()

In [25]:
analyze_parameters(module_types_to_modules[module_type])

Counter()

In [26]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [27]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.activations.SiLUActivation.forward(self, input: torch.Tensor) -> torch.Tensor>: 48})

In [28]:
print(inspect.getsource(next(iter(forwards))))

    def forward(self, input: Tensor) -> Tensor:
        return nn.functional.silu(input)



------

In [29]:
module_type = user_defined_module_types_by_height[2]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeTopKRouter

In [30]:
len(module_types_to_modules[module_type])

48

In [31]:
analyze_child_modules(module_types_to_modules[module_type])

Counter()

In [32]:
analyze_parameters(module_types_to_modules[module_type])

Counter({('weight', True, torch.Size([128, 2048])): 48})

In [33]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [34]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeTopKRouter.forward(self, hidden_states)>: 48})

In [35]:
print(inspect.getsource(next(iter(forwards))))

    def forward(self, hidden_states):
        hidden_states = hidden_states.reshape(-1, self.hidden_dim)
        router_logits = F.linear(hidden_states, self.weight)  # (seq_len, num_experts)
        router_logits = torch.nn.functional.softmax(router_logits, dtype=torch.float, dim=-1)
        router_top_value, router_indices = torch.topk(router_logits, self.top_k, dim=-1)  # (seq_len, top_k)
        if self.norm_topk_prob:
            router_top_value /= router_top_value.sum(dim=-1, keepdim=True)
        router_top_value = router_top_value.to(router_logits.dtype)
        router_scores = router_top_value
        return router_logits, router_scores, router_indices



In [36]:
{self.hidden_dim for self in module_types_to_modules[module_type]}

{2048}

In [37]:
{self.top_k for self in module_types_to_modules[module_type]}

{8}

In [38]:
{self.norm_topk_prob for self in module_types_to_modules[module_type]}

{True}

------

In [39]:
module_type = user_defined_module_types_by_height[3]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRotaryEmbedding

In [40]:
len(module_types_to_modules[module_type])

1

In [41]:
analyze_child_modules(module_types_to_modules[module_type])

Counter()

In [42]:
analyze_parameters(module_types_to_modules[module_type])

Counter()

In [43]:
analyze_buffers(module_types_to_modules[module_type])

Counter({('inv_freq', False, torch.Size([64])): 1,
         ('original_inv_freq', False, torch.Size([64])): 1})

In [44]:
[self.inv_freq for self in module_types_to_modules[module_type]]

[tensor([1.0000e+00, 8.0584e-01, 6.4938e-01, 5.2330e-01, 4.2170e-01, 3.3982e-01,
         2.7384e-01, 2.2067e-01, 1.7783e-01, 1.4330e-01, 1.1548e-01, 9.3057e-02,
         7.4989e-02, 6.0430e-02, 4.8697e-02, 3.9242e-02, 3.1623e-02, 2.5483e-02,
         2.0535e-02, 1.6548e-02, 1.3335e-02, 1.0746e-02, 8.6596e-03, 6.9783e-03,
         5.6234e-03, 4.5316e-03, 3.6517e-03, 2.9427e-03, 2.3714e-03, 1.9110e-03,
         1.5399e-03, 1.2409e-03, 1.0000e-03, 8.0584e-04, 6.4938e-04, 5.2330e-04,
         4.2170e-04, 3.3982e-04, 2.7384e-04, 2.2067e-04, 1.7783e-04, 1.4330e-04,
         1.1548e-04, 9.3057e-05, 7.4989e-05, 6.0430e-05, 4.8697e-05, 3.9242e-05,
         3.1623e-05, 2.5483e-05, 2.0535e-05, 1.6548e-05, 1.3335e-05, 1.0746e-05,
         8.6596e-06, 6.9783e-06, 5.6234e-06, 4.5316e-06, 3.6517e-06, 2.9427e-06,
         2.3714e-06, 1.9110e-06, 1.5399e-06, 1.2409e-06])]

In [45]:
[self.original_inv_freq for self in module_types_to_modules[module_type]]

[tensor([1.0000e+00, 8.0584e-01, 6.4938e-01, 5.2330e-01, 4.2170e-01, 3.3982e-01,
         2.7384e-01, 2.2067e-01, 1.7783e-01, 1.4330e-01, 1.1548e-01, 9.3057e-02,
         7.4989e-02, 6.0430e-02, 4.8697e-02, 3.9242e-02, 3.1623e-02, 2.5483e-02,
         2.0535e-02, 1.6548e-02, 1.3335e-02, 1.0746e-02, 8.6596e-03, 6.9783e-03,
         5.6234e-03, 4.5316e-03, 3.6517e-03, 2.9427e-03, 2.3714e-03, 1.9110e-03,
         1.5399e-03, 1.2409e-03, 1.0000e-03, 8.0584e-04, 6.4938e-04, 5.2330e-04,
         4.2170e-04, 3.3982e-04, 2.7384e-04, 2.2067e-04, 1.7783e-04, 1.4330e-04,
         1.1548e-04, 9.3057e-05, 7.4989e-05, 6.0430e-05, 4.8697e-05, 3.9242e-05,
         3.1623e-05, 2.5483e-05, 2.0535e-05, 1.6548e-05, 1.3335e-05, 1.0746e-05,
         8.6596e-06, 6.9783e-06, 5.6234e-06, 4.5316e-06, 3.6517e-06, 2.9427e-06,
         2.3714e-06, 1.9110e-06, 1.5399e-06, 1.2409e-06])]

In [46]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRotaryEmbedding.forward(self, x, position_ids)>: 1})

In [47]:
print(inspect.getsource(next(iter(forwards))))

    @torch.no_grad()
    @dynamic_rope_update  # power user: used with advanced RoPE types (e.g. dynamic rope)
    def forward(self, x, position_ids):
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()

        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != "mps" else "cpu"
        with maybe_autocast(device_type=device_type, enabled=False):  # Force float32
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos() * self.attention_scaling
            sin = emb.sin() * self.attention_scaling

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)



In [48]:
{self.attention_scaling for self in module_types_to_modules[module_type]}

{1.0}

------

In [49]:
module_type = user_defined_module_types_by_height[4]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeAttention

In [50]:
len(module_types_to_modules[module_type])

48

In [51]:
analyze_child_modules(module_types_to_modules[module_type])

Counter({('q_proj', torch.nn.modules.linear.Linear): 48,
         ('k_proj', torch.nn.modules.linear.Linear): 48,
         ('v_proj', torch.nn.modules.linear.Linear): 48,
         ('o_proj', torch.nn.modules.linear.Linear): 48,
         ('q_norm',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRMSNorm): 48,
         ('k_norm',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRMSNorm): 48})

In [52]:
analyze_parameters(module_types_to_modules[module_type])

Counter()

In [53]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [54]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeAttention.forward(self, hidden_states: torch.Tensor, position_embeddings: tuple[torch.Tensor, torch.Tensor], attention_mask: torch.Tensor | None, past_key_values: transformers.cache_utils.Cache | None = None, cache_position: torch.LongTensor | None = None, **kwargs: Unpack[transformers.modeling_flash_attention_utils.FlashAttentionKwargs]) -> tuple[torch.Tensor, torch.Tensor | None]>: 48})

In [55]:
print(inspect.getsource(next(iter(forwards))))

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: torch.Tensor | None,
        past_key_values: Cache | None = None,
        cache_position: torch.LongTensor | None = None,
        **kwargs: Unpack[FlashAttentionKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
        key_states = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_values is not None:
            # sin and cos are specific to RoPE models; cache_posi

In [56]:
{self.head_dim for self in module_types_to_modules[module_type]}

{128}

In [57]:
{self.config._attn_implementation for self in module_types_to_modules[module_type]}

{'sdpa'}

In [58]:
{self.training for self in module_types_to_modules[module_type]}

{False}

In [59]:
{self.attention_dropout for self in module_types_to_modules[module_type]}

{0.0}

In [60]:
{self.scaling for self in module_types_to_modules[module_type]}

{0.08838834764831845}

In [61]:
{self.sliding_window for self in module_types_to_modules[module_type]}

{None}

In [62]:
apply_rotary_pos_emb = retrieve_global_name(module_type, 'apply_rotary_pos_emb')
apply_rotary_pos_emb

<function transformers.models.qwen3_moe.modeling_qwen3_moe.apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1)>

In [63]:
print(inspect.getsource(apply_rotary_pos_emb))

@use_kernel_func_from_hub("rotary_pos_emb")
def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    """Applies Rotary Position Embedding to the query and key tensors.

    Args:
        q (`torch.Tensor`): The query tensor.
        k (`torch.Tensor`): The key tensor.
        cos (`torch.Tensor`): The cosine part of the rotary embedding.
        sin (`torch.Tensor`): The sine part of the rotary embedding.
        unsqueeze_dim (`int`, *optional*, defaults to 1):
            The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
            cos[position_ids] and sin[position_ids] broadcastable to the

In [64]:
rotate_half = retrieve_global_name(apply_rotary_pos_emb, 'rotate_half')
rotate_half

<function transformers.models.qwen3_moe.modeling_qwen3_moe.rotate_half(x)>

In [65]:
print(inspect.getsource(rotate_half))

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)



In [66]:
ALL_ATTENTION_FUNCTIONS = retrieve_global_name(module_type, 'ALL_ATTENTION_FUNCTIONS')
ALL_ATTENTION_FUNCTIONS

In [67]:
eager_attention_forward = retrieve_global_name(module_type, 'eager_attention_forward')
eager_attention_forward

<function transformers.models.qwen3_moe.modeling_qwen3_moe.eager_attention_forward(module: torch.nn.modules.module.Module, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, attention_mask: torch.Tensor | None, scaling: float, dropout: float = 0.0, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs])>

In [68]:
attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface('sdpa', eager_attention_forward)
attention_interface

<function transformers.integrations.sdpa_attention.sdpa_attention_forward(module: torch.nn.modules.module.Module, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, attention_mask: torch.Tensor | None, dropout: float = 0.0, scaling: float | None = None, is_causal: bool | None = None, **kwargs) -> tuple[torch.Tensor, None]>

In [69]:
print(inspect.getsource(attention_interface))

def sdpa_attention_forward(
    module: torch.nn.Module,
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    attention_mask: torch.Tensor | None,
    dropout: float = 0.0,
    scaling: float | None = None,
    is_causal: bool | None = None,
    **kwargs,
) -> tuple[torch.Tensor, None]:
    if kwargs.get("output_attentions", False):
        logger.warning_once(
            "`sdpa` attention does not support `output_attentions=True`."
            " Please set your attention to `eager` if you want any of these features."
        )
    sdpa_kwargs = {}
    if hasattr(module, "num_key_value_groups"):
        if not use_gqa_in_sdpa(attention_mask, key):
            key = repeat_kv(key, module.num_key_value_groups)
            value = repeat_kv(value, module.num_key_value_groups)
        else:
            sdpa_kwargs = {"enable_gqa": True}

    # Instead of relying on the value set in the module directly, we use the is_causal passed in kwargs if it is presented
    is

In [70]:
{hasattr(module, "num_key_value_groups") for module in module_types_to_modules[module_type]}

{True}

In [71]:
{module.num_key_value_groups for module in module_types_to_modules[module_type]}

{8}

In [72]:
{getattr(module, "is_causal") for module in module_types_to_modules[module_type]}

{True}

------

In [73]:
module_type = user_defined_module_types_by_height[5]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeExperts

In [74]:
len(module_types_to_modules[module_type])

48

In [75]:
analyze_child_modules(module_types_to_modules[module_type])

Counter({('act_fn', transformers.activations.SiLUActivation): 48})

In [76]:
analyze_parameters(module_types_to_modules[module_type])

Counter({('gate_up_proj', True, torch.Size([128, 1536, 2048])): 48,
         ('down_proj', True, torch.Size([128, 2048, 768])): 48})

In [77]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [78]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeExperts.forward(self, hidden_states: torch.Tensor, top_k_index: torch.Tensor, top_k_weights: torch.Tensor) -> torch.Tensor>: 48})

In [79]:
print(inspect.getsource(next(iter(forwards))))

    def forward(
        self,
        hidden_states: torch.Tensor,
        top_k_index: torch.Tensor,
        top_k_weights: torch.Tensor,
    ) -> torch.Tensor:
        final_hidden_states = torch.zeros_like(hidden_states)
        with torch.no_grad():
            expert_mask = torch.nn.functional.one_hot(top_k_index, num_classes=self.num_experts)
            expert_mask = expert_mask.permute(2, 1, 0)
            expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()

        for expert_idx in expert_hit:
            expert_idx = expert_idx[0]
            if expert_idx == self.num_experts:
                continue
            top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
            current_state = hidden_states[token_idx]
            gate, up = nn.functional.linear(current_state, self.gate_up_proj[expert_idx]).chunk(2, dim=-1)
            current_hidden_states = self.act_fn(gate) * up
            current_hidden_states = nn.functional.linear(current_hidd

In [80]:
{self.num_experts for self in module_types_to_modules[module_type]}

{128}

------

In [81]:
module_type = user_defined_module_types_by_height[6]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeSparseMoeBlock

In [82]:
len(module_types_to_modules[module_type])

48

In [83]:
analyze_child_modules(module_types_to_modules[module_type])

Counter({('experts',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeExperts): 48,
         ('gate',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeTopKRouter): 48})

In [84]:
analyze_parameters(module_types_to_modules[module_type])

Counter()

In [85]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [86]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeSparseMoeBlock.forward(self, hidden_states: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]>: 48})

In [87]:
print(inspect.getsource(next(iter(forwards))))

    def forward(self, hidden_states: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size, sequence_length, hidden_dim = hidden_states.shape
        hidden_states_reshaped = hidden_states.view(-1, hidden_dim)
        _, routing_weights, selected_experts = self.gate(hidden_states_reshaped)
        final_hidden_states = self.experts(hidden_states_reshaped, selected_experts, routing_weights)
        return final_hidden_states.reshape(batch_size, sequence_length, hidden_dim)



------

In [88]:
module_type = user_defined_module_types_by_height[7]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeDecoderLayer

In [89]:
len(module_types_to_modules[module_type])

48

In [90]:
analyze_child_modules(module_types_to_modules[module_type])

Counter({('self_attn',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeAttention): 48,
         ('mlp',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeSparseMoeBlock): 48,
         ('input_layernorm',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRMSNorm): 48,
         ('post_attention_layernorm',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRMSNorm): 48})

In [91]:
analyze_parameters(module_types_to_modules[module_type])

Counter()

In [92]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [93]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeDecoderLayer.forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, use_cache: bool | None = False, cache_position: torch.LongTensor | None = None, position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs]) -> torch.Tensor>: 48})

In [94]:
print(inspect.getsource(next(iter(forwards))))

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = False,
        cache_position: torch.LongTensor | None = None,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        # Self Attention
        hidden_states, _ = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            cache_position=cache_position,
            position_embeddings=position_embeddings,
            **kwargs,
        )
        hidden_states = residual + hidden_state

------

In [95]:
module_type = user_defined_module_types_by_height[8]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeModel

In [96]:
len(module_types_to_modules[module_type])

1

In [97]:
analyze_child_modules(module_types_to_modules[module_type])

Counter({('embed_tokens', torch.nn.modules.sparse.Embedding): 1,
         ('layers', torch.nn.modules.container.ModuleList): 1,
         ('norm',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRMSNorm): 1,
         ('rotary_emb',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeRotaryEmbedding): 1})

In [98]:
analyze_parameters(module_types_to_modules[module_type])

Counter()

In [99]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [100]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeModel.forward(self, input_ids: torch.LongTensor | None = None, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, inputs_embeds: torch.FloatTensor | None = None, use_cache: bool | None = None, cache_position: torch.LongTensor | None = None, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs]) -> transformers.modeling_outputs.MoeModelOutputWithPast>: 1})

In [101]:
print(inspect.getsource(next(iter(forwards))))

    @merge_with_config_defaults
    @capture_outputs
    @auto_docstring
    def forward(
        self,
        input_ids: torch.LongTensor | None = None,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        inputs_embeds: torch.FloatTensor | None = None,
        use_cache: bool | None = None,
        cache_position: torch.LongTensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> MoeModelOutputWithPast:
        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("You must specify exactly one of input_ids or inputs_embeds")

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache(config=self.config)

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        if cache_position is None:
            past_seen_tokens = past_key_values.get_seq_length() if p

In [102]:
{self.config.sliding_window for self in module_types_to_modules[module_type]}

{None}

------

In [103]:
module_type = user_defined_module_types_by_height[9]
module_type

transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeForCausalLM

In [104]:
len(module_types_to_modules[module_type])

1

In [105]:
analyze_child_modules(module_types_to_modules[module_type])

Counter({('model',
          transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeModel): 1,
         ('lm_head', torch.nn.modules.linear.Linear): 1})

In [106]:
analyze_parameters(module_types_to_modules[module_type])

Counter()

In [107]:
analyze_buffers(module_types_to_modules[module_type])

Counter()

In [108]:
forwards = analyze_forwards(module_types_to_modules[module_type])
forwards

Counter({<function transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeForCausalLM.forward(self, input_ids: torch.LongTensor | None = None, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, inputs_embeds: torch.FloatTensor | None = None, labels: torch.LongTensor | None = None, use_cache: bool | None = None, output_router_logits: bool | None = None, cache_position: torch.LongTensor | None = None, logits_to_keep: int | torch.Tensor = 0, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs]) -> transformers.modeling_outputs.MoeCausalLMOutputWithPast>: 1})

In [109]:
print(inspect.getsource(next(iter(forwards))))

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.LongTensor | None = None,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        inputs_embeds: torch.FloatTensor | None = None,
        labels: torch.LongTensor | None = None,
        use_cache: bool | None = None,
        output_router_logits: bool | None = None,
        cache_position: torch.LongTensor | None = None,
        logits_to_keep: int | torch.Tensor = 0,
        **kwargs: Unpack[TransformersKwargs],
    ) -> MoeCausalLMOutputWithPast:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,
            config.vocab_size]` or -100 (see `input_ids` docstring). Tokens with indices set to `-100` are ignored
            (masked),